# Part 2b — Analyze SPINE output files

SPINE output files (`.h5`) contain what the reconstruction found: **particles** and
**interactions**, both reconstructed (`reco_`) and, for simulation, true (`truth_`).
In this notebook you will:
1. peek inside a file with `h5py`;
2. load events as Python objects and read their properties;
3. draw an event display;
4. build pandas tables and histograms over all events;
5. use reco ↔ truth **matching** to measure the efficiency and purity of your own selection;
6. do the same with a **lite** file, and export everything to CSV.

**Session needed:** any. CPU only (partition `milano`, 0 GPUs) is fine, and it even runs
on a laptop with `pip install "spine[viz]"`.

> **Terminal equivalent** for the CSV export at the end:
> ```bash
> spine_container.sh spine -c $SPINE_TUTORIAL/configs/ana_save_csv.yaml -s my_output.h5 --log-dir $WORKDIR/ana
> ```

## 0. Setup and choose the input file

In [ ]:
import sys; sys.path.insert(0, "..")
import os
from pathlib import Path
import numpy as np
import pandas as pd
import tutorial_env as te
env = te.setup()

OUT = Path(env["WORKDIR"]) / "out"
OUT.mkdir(parents=True, exist_ok=True)

# Use the files you made in Part 1 if they exist; otherwise download a ready-made one.
candidates = [OUT / "protodune-sp_small_full.h5", OUT / "generic_small_full.h5"]
full_file = next((p for p in candidates if p.exists()), None) or te.fetch("generic_small_spine.h5", "reco")
print("\nFULL file used in this notebook:", full_file)

💡 To analyze your own production instead, set `full_file = "/path/to/your_file.h5"`.
`file_keys` (used below) also accepts a list of files or a pattern such as `"/path/*.h5"`.

## 1. Peek inside with `h5py`

An HDF5 file is like a folder of tables. `events` has one row per event and says where that
event's objects sit in each of the other tables.

In [ ]:
import h5py

with h5py.File(full_file, "r") as f:
    print("products :", list(f.keys()))
    print("events   :", len(f["events"]))
    print("made with SPINE", f["info"].attrs.get("spine_version", f["info"].attrs.get("version")))

## 2. Load events as objects: `Driver` + `build`

Reading raw tables is painful. SPINE's `Driver` reads the file **and rebuilds** proper Python
objects (`RecoParticle`, `TruthInteraction`, …), reconnecting each with its voxels and its
best reco ↔ truth match. It only needs a tiny config, [`../configs/read_spine_output.yaml`](../configs/read_spine_output.yaml):

```yaml
io:    {reader: {provider: hdf5, file_keys: <your file>}}
build: {mode: both, units: cm, fragments: false, particles: true, interactions: true}
```

In [ ]:
from spine.config import load_config_file
from spine.driver import Driver

def make_driver(path, lite=False):
    """Driver that reads a SPINE output file (full or lite) and rebuilds objects."""
    cfg_name = "read_spine_output_lite.yaml" if lite else "read_spine_output.yaml"
    cfg = load_config_file(os.path.join(env["SPINE_TUTORIAL"], "configs", cfg_name))
    cfg["io"]["reader"]["file_keys"] = str(path)
    cfg["base"]["log_dir"] = os.path.join(env["WORKDIR"], "logs")
    return Driver(cfg)

driver = make_driver(full_file)
print(len(driver), "entries")

`driver.process(entry=i)` returns a dictionary for event `i`:

In [ ]:
data = driver.process(entry=0)
for key in sorted(data):
    value = data[key]
    size = f"{len(value)} items" if hasattr(value, "__len__") and not isinstance(value, str) else value
    print(f"{key:<34} {size}")

Key entries:
- `reco_particles`, `truth_particles`, `reco_interactions`, `truth_interactions`: lists of objects.
- `points`, `depositions`: every voxel (full files only).
- `particle_matches_r2t`: pairs `(reco particle, best-matching true particle or None)`. `t2r` is the other direction, and `interaction_matches_*` do the same for interactions.

## 3. Particles and interactions

Each object has dozens of attributes. Numbers like `pid` and `shape` are codes; the
`*_LABELS` dictionaries translate them.

In [ ]:
from spine.constants import PID_LABELS, SHAPE_LABELS
print("PID codes  :", PID_LABELS)
print("shape codes:", SHAPE_LABELS)

print(f"\n{'id':>3} {'inter':>5} {'shape':<7} {'PID':<9} {'primary':<7} {'KE [MeV]':>9} {'length [cm]':>11}  matched-to-truth")
for p in data["reco_particles"]:
    print(f"{p.id:>3} {p.interaction_id:>5} {SHAPE_LABELS[p.shape]:<7} {PID_LABELS[p.pid]:<9} "
          f"{str(bool(p.is_primary)):<7} {p.ke:9.1f} {p.length:11.1f}  {p.match_ids.tolist()}")

`ke` is SPINE's best kinetic-energy estimate. It picks range (`csda_ke`) for contained tracks,
multiple scattering (`mcs_ke`) for exiting tracks, and calorimetry (`calo_ke`) for showers.
`-1` means "not computed". `length` is only defined for tracks.

In [ ]:
for inter in data["reco_interactions"]:
    print(f"reco interaction {inter.id}: topology '{inter.topology}', "
          f"{inter.num_particles} particles ({inter.num_primary_particles} primary), "
          f"vertex {np.round(inter.vertex, 1)} cm, fiducial={bool(inter.is_fiducial)}, "
          f"contained={bool(inter.is_contained)}")
print()
for inter in data["truth_interactions"]:
    print(f"true interaction {inter.id}: topology '{inter.topology}', "
          f"primaries [γ, e, μ, π, p, K] = {inter.primary_particle_counts.tolist()}")

## 4. Event display

`Drawer` makes interactive 3D plots. With the default `draw_mode="both"` you get reco on the
left and truth on the right. `attr` lists the values shown when you hover over an object, and
`color_attr` (which must be one of them) picks what sets the colours.

In [ ]:
from spine.vis import Drawer

drawer = Drawer(data)          # reco | truth side by side
fig = drawer.get("particles", attr=["shape", "pid", "ke", "is_primary"], color_attr="shape",
                 draw_end_points=True)
fig.show()

In [ ]:
fig = drawer.get("interactions", attr=["id", "topology", "is_fiducial"], draw_vertices=True)
fig.show()

> If a figure is blank, run `import plotly.io as pio; pio.renderers.default = "iframe"` and re-run the cell.

**Try:** `color_attr="pid"`, or add `draw_raw=True` to overlay the raw input charge.

## 5. Tables over all events

Analyses loop over every entry and collect what they need into a table. `scalar_dict()` turns
an object into a flat dictionary, ready for pandas.

In [ ]:
PARTICLE_COLS = ["id", "interaction_id", "shape", "pid", "is_primary", "ke", "length",
                 "is_contained", "is_matched"]
INTERACTION_COLS = ["id", "num_particles", "num_primary_particles", "topology",
                    "is_fiducial", "is_contained", "is_matched"]

def collect(driver, key, cols):
    """One row per object of type `key` (e.g. 'reco_particles') across all entries."""
    rows = []
    for entry in range(len(driver)):
        event = driver.process(entry=entry)
        for obj in event[key]:
            row = obj.scalar_dict(attrs=cols)
            row["entry"] = entry
            rows.append(row)
    return pd.DataFrame(rows)

reco_p = collect(driver, "reco_particles", PARTICLE_COLS)
reco_p["pid_name"] = reco_p["pid"].map(PID_LABELS)
reco_p.head(10)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Number of reconstructed particles of each type
reco_p["pid_name"].value_counts().plot.bar(ax=axes[0], color="steelblue")
axes[0].set_title("Reconstructed particles by PID"); axes[0].set_ylabel("count")

# Kinetic energy of primary particles, per type (only those with a valid KE)
prim = reco_p[(reco_p["is_primary"] == True) & (reco_p["ke"] > 0) & (reco_p["ke"] < 3000)]
for name, group in prim.groupby("pid_name"):
    axes[1].hist(group["ke"], bins=20, range=(0, 2000), histtype="step", lw=2, label=name)
axes[1].set_xlabel("kinetic energy [MeV]"); axes[1].set_title("Primary particles"); axes[1].legend()
plt.tight_layout(); plt.show()

With the small example files the histograms are sparse. Point `full_file` at a real
production (Part 1, section 5) and the same code gives proper distributions.

## 6. Reco ↔ truth matching: efficiency and purity of a selection

In simulation, every reco interaction is matched to the true interaction it overlaps with
most, and vice versa. SPINE stores the pairs:
- `interaction_matches_t2r`: list of `(true interaction, best reco interaction or None)`
- `interaction_matches_r2t`: list of `(reco interaction, best true interaction or None)`

A **selection** is a yes/no question asked of each interaction ("is this a contained νμ CC
candidate?"). Asking the *same* question of truth and reco gives you two numbers:
- **efficiency**: of the true interactions that pass, how many does the reconstruction also select?
- **purity**: of the reco interactions you select, how many really pass according to truth?

### ✏️ Your turn
Open [`selection.py`](selection.py) (next to this notebook) and write `select_interaction(inter)`.
The file lists the available attributes. Save it, then run the cells below; the notebook
reloads your file automatically.

In [ ]:
%load_ext autoreload
%autoreload 2
from selection import select_interaction

In [ ]:
def efficiency_purity(driver, select):
    """Count selected interactions in truth and reco, using the stored matches."""
    n_true_sel = n_true_sel_and_reco_sel = 0
    n_reco_sel = n_reco_sel_and_true_sel = 0
    for entry in range(len(driver)):
        event = driver.process(entry=entry)
        for true_inter, reco_inter in event["interaction_matches_t2r"]:
            if select(true_inter):
                n_true_sel += 1
                if reco_inter is not None and select(reco_inter):
                    n_true_sel_and_reco_sel += 1
        for reco_inter, true_inter in event["interaction_matches_r2t"]:
            if select(reco_inter):
                n_reco_sel += 1
                if true_inter is not None and select(true_inter):
                    n_reco_sel_and_true_sel += 1
    eff = n_true_sel_and_reco_sel / n_true_sel if n_true_sel else float("nan")
    pur = n_reco_sel_and_true_sel / n_reco_sel if n_reco_sel else float("nan")
    print(f"true interactions passing : {n_true_sel}")
    print(f"reco interactions passing : {n_reco_sel}")
    print(f"efficiency = {n_true_sel_and_reco_sel}/{n_true_sel} = {eff:.2f}")
    print(f"purity     = {n_reco_sel_and_true_sel}/{n_reco_sel} = {pur:.2f}")
    return eff, pur

efficiency_purity(driver, select_interaction);

**Interpreting it:** low efficiency means the reconstruction often fails to recognise true
signal events. Common causes are a missed proton or a split track. Low purity means
background sneaks in. Tighten or loosen your selection and re-run to see the trade-off.
With only a handful of events the numbers jump around; use a real production for meaningful values.

## 7. Lite files

Lite files contain the same particle and interaction objects but **no voxels**. They need
`build.lite: true` ([`../configs/read_spine_output_lite.yaml`](../configs/read_spine_output_lite.yaml)).
If you don't have one from Part 1, make one now from the full file with spine-prod's
`litify` config. It runs on CPU in seconds:

In [ ]:
lite_file = Path(str(full_file).replace("_full.h5", "_lite.h5"))
if not lite_file.exists() or lite_file == Path(full_file):
    lite_file = OUT / (Path(full_file).stem + "_lite.h5")
    te.sh(f"spine -c infer/common/litify.yaml -s {full_file} -o {lite_file} "
          f"--log-dir {env['WORKDIR']}/logs")
print("LITE file:", lite_file, f"({lite_file.stat().st_size / 1e6:.2f} MB vs "
      f"{Path(full_file).stat().st_size / 1e6:.2f} MB full)")

In [ ]:
lite_driver = make_driver(lite_file, lite=True)
lite_data = lite_driver.process(entry=0)
p = lite_data["reco_particles"][0]
print("same physics quantities:", PID_LABELS[p.pid], f"KE={p.ke:.1f} MeV")
print("but no voxels          : points array shape", p.points.shape)

# Same table code as before works unchanged
collect(lite_driver, "reco_interactions", INTERACTION_COLS).head()

In [ ]:
# Same efficiency/purity, from the lite file
efficiency_purity(lite_driver, select_interaction);

In [ ]:
# Event display without voxels: objects are drawn as simple markers/lines
Drawer(lite_data, lite=True).get("particles", attr=["pid", "ke"], color_attr="pid").show()

## 8. Export to CSV (no Python needed afterwards)

SPINE's `save` analysis script writes one CSV per object type, with matched-object columns
added. This is the usual way to hand results to ROOT, Excel or another framework. The config is
[`../configs/ana_save_csv.yaml`](../configs/ana_save_csv.yaml).

In [ ]:
ana_dir = Path(env["WORKDIR"]) / "ana"
te.sh(f"spine -c $SPINE_TUTORIAL/configs/ana_save_csv.yaml -s {full_file} --log-dir {ana_dir}")
print(sorted(p.name for p in ana_dir.glob("*.csv")))

In [ ]:
csv = pd.read_csv(ana_dir / "save_reco_interactions.csv")
print(len(csv.columns), "columns, e.g.", list(csv.columns[:12]))
csv.head()

For a whole production, run the same config as a CPU batch job over the list of outputs (from an `ssh neutrino` terminal):
```bash
cd $WORKDIR
$SPINE_PROD_BASEDIR/submit.py --config $SPINE_TUTORIAL/configs/ana_save_csv.yaml \
    --source-list runs/<your run>/latest/outputs.txt --profile s3df_milano -A $SPINE_ACCOUNT
```
For lite inputs add `--set build.lite=true`.

## Summary

| Task | How |
|---|---|
| quick look | `h5py.File(path)` |
| objects + matches | `Driver(cfg)` with [`read_spine_output.yaml`](../configs/read_spine_output.yaml), then `.process(entry=i)` |
| event display | `Drawer(data).get("particles", color_attr="shape")` |
| table | `obj.scalar_dict(attrs=[...])` → pandas |
| efficiency / purity | loop over `interaction_matches_t2r` / `_r2t` |
| lite files | `read_spine_output_lite.yaml` (`build.lite: true`), `Drawer(..., lite=True)` |
| CSV | `spine -c ana_save_csv.yaml -s file.h5 --log-dir DIR` |

Next: **[Part 3 — training](../03_training/README.md)**.